# Expanded Q4 2025 AI/Tech Earnings Evaluation

This notebook runs the full 55-name Q4 2025 AI/Tech earnings evaluation in one clean flow:

1. Load repo settings and cache paths
2. Build the post-earnings price evaluation table
3. Load or refresh cached agent packets
4. Run missing or forced CIO workflows incrementally
5. Rebuild `cio_eval` from cached CIO workflows
6. Compare CIO against single agents and GPT/Gemini/Claude external baselines

Default settings reuse cache and avoid expensive LLM reruns. Change the flags in the config cell when you intentionally want to refresh outputs.


In [37]:
import os
import sys
import importlib
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display, Markdown

CURRENT_DIR = Path.cwd().resolve()
REPO_ROOT = CURRENT_DIR if (CURRENT_DIR / "pyproject.toml").exists() else CURRENT_DIR.parent
SRC_PATH = REPO_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

load_dotenv(REPO_ROOT / ".env", override=True)

from openclam.evaluation import q4_earnings_cache as q4
from openclam.agents.news_macro import news_macro_agent

q4 = importlib.reload(q4)

print(f"Repo root: {REPO_ROOT}")
print(f"Loaded q4 module: {q4.__file__}")
print(f"VERTEX_PROJECT loaded: {bool(os.getenv('VERTEX_PROJECT') or os.getenv('GOOGLE_CLOUD_PROJECT'))}")
print(f"OPENAI_API_KEY loaded: {bool(os.getenv('OPENAI_API_KEY'))}")


Repo root: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory
Loaded q4 module: /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/src/openclam/evaluation/q4_earnings_cache.py
VERTEX_PROJECT loaded: True
OPENAI_API_KEY loaded: False


## Configuration

The default universe is the full 55-name Q4 2025 AI/Tech set defined in `q4_earnings_cache.py`.

Set `RUN_AGENT_REFRESH=True` only if you want to regenerate all base agent packets. Set `REFRESH_MARKET_TECHNICAL=True` if you updated the Market/Technical agent and want to refresh that agent plus packets. Set `RUN_CIO=True` to run missing CIO workflows; set `FORCE_RERUN_CIO=True` to overwrite existing CIO workflows.


In [38]:
CACHE_ROOT = REPO_ROOT / q4.DEFAULT_CACHE_ROOT
CACHE_ROOT.mkdir(parents=True, exist_ok=True)

PRE_TRADING_DAYS = 7
POST_TRADING_DAYS = 7
LONG_POST_TRADING_DAYS = 30
NEUTRAL_BAND = 0.02

VERTEX_PROJECT = os.getenv("VERTEX_PROJECT") or os.getenv("GOOGLE_CLOUD_PROJECT")
VERTEX_LOCATION = os.getenv("VERTEX_LOCATION", "us-central1")
VERTEX_MODEL = os.getenv("VERTEX_MODEL", "gemini-2.5-flash")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
NEWS_MODEL = os.getenv("NEWS_MODEL", "gpt-5.4-nano")
LLM_PROVIDER = "vertex" if VERTEX_PROJECT else "openai"

RUN_AGENT_REFRESH = False
FORCE_RERUN_AGENTS = False
REFRESH_MARKET_TECHNICAL = False
RUN_CIO = False
FORCE_RERUN_CIO = False

# Optional: limit to a subset for quick demos. Keep None for the full 55-name universe.
SELECTED_TICKERS = None

earnings_df = q4.q4_2025_combined_cio_advantage_df(SELECTED_TICKERS)
print("Universe size:", len(earnings_df))
display(earnings_df)


Universe size: 55


,ticker,company,earnings_date,bucket
0,AAPL,Apple,2026-01-29,mega_cap_platform
1,MSFT,Microsoft,2026-01-28,mega_cap_platform
2,GOOGL,Alphabet,2026-02-03,mega_cap_platform
3,AMZN,Amazon,2026-02-05,mega_cap_platform
4,META,Meta Platforms,2026-01-28,mega_cap_platform
5,TSLA,Tesla,2026-01-28,mega_cap_platform
6,NVDA,Nvidia,2026-02-25,mega_cap_platform
7,AMD,Advanced Micro Devices,2026-02-03,ai_semis
8,AVGO,Broadcom,2026-03-05,ai_semis
9,MU,Micron Technology,2025-12-17,ai_semis


## Build Price Evaluation Table

Returns are measured after the earnings-date close. Short horizon is 7 trading days; long horizon is 30 trading days; abnormal returns are measured versus QQQ.


In [39]:
summary_df, paths_df = q4.build_price_tables(
    earnings_df=earnings_df,
    cache_root=CACHE_ROOT,
    pre_trading_days=PRE_TRADING_DAYS,
    post_trading_days=POST_TRADING_DAYS,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
)

print("Price summary rows:", len(summary_df))
news_macro_agent.format_return_columns(summary_df)


Price summary rows: 55


,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,spy_post_30d_return,abnormal_30d_vs_spy,qqq_post_7d_return,abnormal_vs_qqq,qqq_post_30d_return,abnormal_30d_vs_qqq,realized_direction_vs_qqq,realized_30d_direction_vs_qqq,long_horizon_trading_days,bucket
0,AAPL,Apple,2026-01-29,4.69%,0.46%,4.34%,6.43%,-3.07%,11.42%,1.48%,...,-4.57%,1.51%,-2.40%,8.83%,-5.67%,2.60%,up,up,30,mega_cap_platform
1,MSFT,Microsoft,2026-01-28,4.73%,-9.99%,-12.10%,-16.71%,-16.37%,-12.77%,-12.41%,...,-4.22%,-12.15%,-3.72%,-12.99%,-5.68%,-10.69%,down,down,30,mega_cap_platform
2,GOOGL,Alphabet,2026-02-03,3.59%,-1.96%,-4.96%,-9.04%,-9.36%,-5.77%,-6.11%,...,-4.08%,-5.29%,-2.58%,-6.46%,-3.51%,-5.86%,down,down,30,mega_cap_platform
3,AMZN,Amazon,2026-02-05,-8.99%,-5.55%,-7.06%,-9.67%,-7.78%,-17.79%,-16.07%,...,-4.03%,-3.75%,0.72%,-10.39%,-2.51%,-5.27%,down,down,30,mega_cap_platform
4,META,Meta Platforms,2026-01-28,7.82%,10.40%,5.63%,-1.09%,-4.57%,6.64%,2.89%,...,-4.22%,-0.35%,-3.72%,2.64%,-5.68%,1.11%,up,up,30,mega_cap_platform
5,TSLA,Tesla,2026-01-28,-1.38%,-3.45%,-2.24%,-4.72%,-8.45%,-6.03%,-9.71%,...,-4.22%,-4.23%,-3.72%,-0.99%,-5.68%,-2.77%,down,down,30,mega_cap_platform
6,NVDA,Nvidia,2026-02-25,6.97%,-5.46%,-6.69%,-9.07%,-5.95%,-2.73%,0.61%,...,-1.64%,-4.31%,-2.75%,-6.33%,-0.93%,-5.02%,down,down,30,mega_cap_platform
7,AMD,Advanced Micro Devices,2026-02-03,-6.77%,-17.31%,-13.91%,-14.94%,-17.62%,-20.69%,-23.19%,...,-4.08%,-13.54%,-2.58%,-12.36%,-3.51%,-14.11%,down,down,30,ai_semis
8,AVGO,Broadcom,2026-03-05,2.24%,-0.69%,2.95%,-2.36%,22.42%,-0.18%,25.16%,...,4.52%,17.91%,-1.40%,-0.96%,6.69%,15.73%,down,up,30,ai_semis
9,MU,Micron Technology,2025-12-17,-8.67%,10.21%,22.65%,30.58%,94.21%,19.26%,77.38%,...,3.88%,90.33%,3.54%,27.04%,4.42%,89.79%,up,up,30,ai_semis


## Load or Refresh Agent Packets

This cell loads cached packets by default. If you changed agent code, turn on the relevant refresh flag in the configuration cell.


In [40]:
if RUN_AGENT_REFRESH:
    packets_by_ticker, errors_by_ticker = q4.run_q4_2025_universe_agents(
        earnings_df=earnings_df,
        cache_root=CACHE_ROOT,
        force=FORCE_RERUN_AGENTS,
        lookback_days=14,
        max_news=10,
        llm_provider=LLM_PROVIDER,
        vertex_project=VERTEX_PROJECT,
        vertex_location=VERTEX_LOCATION,
        vertex_model=VERTEX_MODEL,
        openai_model=OPENAI_MODEL,
        news_model=NEWS_MODEL,
    )
elif REFRESH_MARKET_TECHNICAL:
    packets_by_ticker = q4.rerun_cached_market_technical_agent(
        earnings_df,
        cache_root=CACHE_ROOT,
        vertex_project=VERTEX_PROJECT,
        vertex_location=VERTEX_LOCATION,
        vertex_model=VERTEX_MODEL,
        openai_model=OPENAI_MODEL,
    )
    errors_by_ticker = {}
else:
    packets_by_ticker = q4.load_cached_packets_by_ticker(CACHE_ROOT)
    errors_by_ticker = {}

expected_tickers = set(summary_df["ticker"].str.upper())
available_tickers = set(packets_by_ticker)
print("Expected tickers:", len(expected_tickers))
print("Cached packet tickers:", len(available_tickers & expected_tickers))
print("Missing packet tickers:", sorted(expected_tickers - available_tickers))
errors_by_ticker


Expected tickers: 55
Cached packet tickers: 55
Missing packet tickers: []


{}

## CIO Workflows

This cell runs CIO workflows incrementally only when requested, then always rebuilds `cio_eval` from cached per-ticker CIO JSON files. Incremental saving means interrupted runs keep completed tickers.


In [42]:
cio_dir = CACHE_ROOT / "cio"
cio_dir.mkdir(parents=True, exist_ok=True)
expected_tickers = list(summary_df["ticker"].str.upper())
cached_cio_tickers = {path.stem.upper() for path in cio_dir.glob("*.json")}
missing_cio_tickers = [ticker for ticker in expected_tickers if ticker not in cached_cio_tickers]

if FORCE_RERUN_CIO:
    tickers_to_run = expected_tickers
elif RUN_CIO:
    tickers_to_run = missing_cio_tickers
else:
    tickers_to_run = []

print("Expected CIO tickers:", len(expected_tickers))
print("Cached CIO tickers:", len(cached_cio_tickers & set(expected_tickers)))
print("Missing CIO tickers:", missing_cio_tickers)
print("Tickers selected for CIO run:", tickers_to_run)

incremental_cio_evals = []
incremental_cio_results = {}

for ticker in tickers_to_run:
    print("" + "=" * 80)
    print(f"Running CIO for {ticker}...")
    print("=" * 80)

    one_summary_df = summary_df[summary_df["ticker"].str.upper() == ticker].copy()
    one_packets = {ticker: packets_by_ticker.get(ticker, [])}

    if not one_packets[ticker]:
        print(f"Skipped {ticker}: no packets found.")
        continue

    try:
        one_cio_eval, one_cio_results, _ = q4.run_cached_cio_eval(
            one_summary_df,
            one_packets,
            cache_root=CACHE_ROOT,
            long_post_trading_days=LONG_POST_TRADING_DAYS,
            neutral_band=NEUTRAL_BAND,
            use_llm_debate=True,
            use_llm_decision=True,
            llm_provider=LLM_PROVIDER,
            debate_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
            decision_model=VERTEX_MODEL if VERTEX_PROJECT else OPENAI_MODEL,
            vertex_project=VERTEX_PROJECT,
            vertex_location=VERTEX_LOCATION,
        )
        incremental_cio_evals.append(one_cio_eval)
        incremental_cio_results[ticker] = one_cio_results.get(ticker)
        print(f"Saved CIO result for {ticker}: {cio_dir / f'{ticker}.json'}")
    except Exception as exc:
        print(f"Failed CIO for {ticker}: {repr(exc)}")

cio_eval, cio_results, cio_summary = q4.rebuild_cio_eval_from_cache(
    summary_df,
    cache_root=CACHE_ROOT,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    neutral_band=NEUTRAL_BAND,
    output_name="cio_eval_full_55" if len(summary_df) >= 50 else "cio_eval_selected",
)

print(cio_summary)
news_macro_agent.format_return_columns(cio_eval)


Expected CIO tickers: 55
Cached CIO tickers: 55
Missing CIO tickers: []
Tickers selected for CIO run: []
{'overall': {'cases': 55, 'cio_ready_cases': 55, 'debate_trigger_rate': 0.509090909090909, 'cio_short_direction_match_evaluable': 55, 'cio_short_direction_match_matched': 23, 'cio_short_direction_match_accuracy': 0.41818181818181815, 'cio_long_direction_match_evaluable': 55, 'cio_long_direction_match_matched': 30, 'cio_long_direction_match_accuracy': 0.5454545454545454}, 'by_bucket': {'ai_infrastructure': {'cases': 9, 'debate_trigger_rate': 0.5555555555555556, 'short_accuracy': 0.5555555555555556, 'long_accuracy': 0.5555555555555556, 'avg_confidence': 0.7833333333333333}, 'ai_semis': {'cases': 13, 'debate_trigger_rate': 0.3076923076923077, 'short_accuracy': 0.5384615384615384, 'long_accuracy': 0.6153846153846154, 'avg_confidence': 0.8148461538461539}, 'data_center_operator': {'cases': 2, 'debate_trigger_rate': 0.5, 'short_accuracy': 0.0, 'long_accuracy': 0.5, 'avg_confidence': 0.575

,ticker,company,earnings_date,pre_7d_return,post_1d_return,post_3d_return,post_7d_return,post_30d_return,full_window_return,full_30d_window_return,...,cio_confidence,cio_debate_triggered,cio_debate_conflict_level,cio_routing_bucket,cio_routing_rationale,cio_reason,cio_short_direction_match,cio_short_direction_match_reason,cio_long_direction_match,cio_long_direction_match_reason
0,AAPL,Apple,2026-01-29,4.69%,0.46%,4.34%,6.43%,-3.07%,11.42%,1.48%,...,0.850,True,high,None,None,The committee reached a unanimous Bullish shor...,True,matched,False,neutral missed: abnormal return moved outside ...
1,MSFT,Microsoft,2026-01-28,4.73%,-9.99%,-12.10%,-16.71%,-16.37%,-12.77%,-12.41%,...,0.750,False,low,None,None,The decision aligns with the deterministic gua...,True,matched,False,missed
2,GOOGL,Alphabet,2026-02-03,3.59%,-1.96%,-4.96%,-9.04%,-9.36%,-5.77%,-6.11%,...,0.700,False,low,None,None,The final decision is to adopt a Bearish short...,True,matched,False,missed
3,AMZN,Amazon,2026-02-05,-8.99%,-5.55%,-7.06%,-9.67%,-7.78%,-17.79%,-16.07%,...,0.850,True,high,None,None,The committee reached a high-conviction consen...,True,matched,False,missed
4,META,Meta Platforms,2026-01-28,7.82%,10.40%,5.63%,-1.09%,-4.57%,6.64%,2.89%,...,0.800,True,high,None,None,The final decision is an override of the initi...,False,missed,True,matched
5,TSLA,Tesla,2026-01-28,-1.38%,-3.45%,-2.24%,-4.72%,-8.45%,-6.03%,-9.71%,...,0.850,True,high,None,None,The decision is driven by the unanimous post-d...,True,matched,True,matched
6,NVDA,Nvidia,2026-02-25,6.97%,-5.46%,-6.69%,-9.07%,-5.95%,-2.73%,0.61%,...,0.650,True,high,None,None,CIO Override: I am overriding the deterministi...,False,missed,False,neutral missed: abnormal return moved outside ...
7,AMD,Advanced Micro Devices,2026-02-03,-6.77%,-17.31%,-13.91%,-14.94%,-17.62%,-20.69%,-23.19%,...,0.850,True,high,None,None,I am overriding the deterministic guardrail's ...,True,matched,False,missed
8,AVGO,Broadcom,2026-03-05,2.24%,-0.69%,2.95%,-2.36%,22.42%,-0.18%,25.16%,...,0.800,False,low,None,None,The committee is overriding the guardrail's 'S...,False,missed,True,matched
9,MU,Micron Technology,2025-12-17,-8.67%,10.21%,22.65%,30.58%,94.21%,19.26%,77.38%,...,0.872,False,low,None,None,The committee reached a high-conviction bullis...,True,matched,True,matched


## Optional CIO Inspection

Use this cell to inspect one ticker's saved agent packets and debate trace.


In [43]:
INSPECT_TICKER = "NVDA"

cached = q4.load_cached_ticker(INSPECT_TICKER, CACHE_ROOT)
print("Saved files:")
for key, value in q4.cached_ticker_paths(INSPECT_TICKER, CACHE_ROOT).items():
    print(key, value)

workflow = cio_results.get(INSPECT_TICKER) or q4.load_json(CACHE_ROOT / "cio" / f"{INSPECT_TICKER}.json")
agent_summary = pd.DataFrame(workflow["synthesis"].get("agent_view_summary", []))
debate_summary = pd.DataFrame(workflow["debate"].get("debate_responses", []))

display(Markdown(f"### {INSPECT_TICKER} Agent View Summary"))
display(agent_summary)

display(Markdown(f"### {INSPECT_TICKER} Debate Responses"))
display(debate_summary)

display(Markdown(f"### {INSPECT_TICKER} Final CIO Decision"))
display(workflow["final_decision"])


Saved files:
news_macro /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/news_macro/NVDA.json
market_technical /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/market_technical/NVDA.json
fundamental /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/fundamental/NVDA.json
packets /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/packets/NVDA.json
cio /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/cio/NVDA.json
context /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/contexts/NVDA.json
errors /Users/zhang/Github/OpenClam_Multi-Agent_Investment_Advisory/data/agent_outputs/q4_2025_ai_tech/errors/NVDA.json


### NVDA Agent View Summary

,agent_name,short_term_stance,long_term_stance,confidence,rationale
0,news_macro,Bullish,Bearish,0.65,The short-term stance is Bullish due to a dire...
1,fundamental,Neutral,Bullish,0.60,NVIDIA reported an exceptionally strong quarte...
2,market_technical,Neutral,Bullish,0.70,The primary tension is between the established...


### NVDA Debate Responses

,response_type,agreement,disagreement,revised_short_term_stance,revised_long_term_stance,revised_confidence,revision_rationale,evidence_needed
0,revision,{'fundamental': 'The fundamental agent's data ...,{'fundamental': 'I disagree with the fundament...,Neutral,Bearish,0.60,I am revising my short-term stance from 'Bulli...,[Future earnings reports from AMD to quantify ...
1,revision,{'news_macro': 'I agree with the news_macro ag...,{'news_macro': 'I disagree with the news_macro...,Bullish,Bullish,0.80,My original Neutral short-term stance and mode...,[Quantification of the financial impact of the...
2,revision,[I agree with the news_macro agent that the re...,[I assign a lower weight to the news_macro age...,Bullish,Bullish,0.85,NaN,[The official earnings release detailing speci...


### NVDA Final CIO Decision

{'ticker': 'NVDA',
 'company': 'Nvidia',
 'final_short_term_stance': 'Bullish',
 'final_long_term_stance': 'Neutral',
 'investment_action': 'Hold / Monitor',
 'position_size_hint': 'Small',
 'confidence': 0.65,
 'core_thesis': "The short-term outlook is bullish, driven by an exceptional earnings and guidance beat that should act as a catalyst to resolve the recent price consolidation to the upside. However, the long-term view is downgraded to neutral due to a significant conflict between stellar current fundamentals and mounting, unpriced, forward-looking risks. While Nvidia's dominance is undeniable today, credible threats from competition (e.g., AMD's win with Meta) and geopolitical supply chain risks (TSM dependency) create significant uncertainty about the sustainability of its current market share and valuation premium.",
 'why_now': "The immediate catalyst is the forecast-beating earnings report, which has temporarily allayed fears of an AI slowdown. The stock's recent consolidat

## Final Comparison: CIO vs Agents and External Baselines

This cell compares CIO against the three single agents and any external baseline CSVs saved under:

- `baseline_gpt.csv`
- `baseline_gemini.csv`
- `baseline_claude.csv`


In [45]:
comparison_bundle = q4.compare_with_external_baselines(
    cio_eval,
    packets_by_ticker,
    cache_root=CACHE_ROOT,
    long_post_trading_days=LONG_POST_TRADING_DAYS,
    neutral_band=NEUTRAL_BAND,
)

comparison_eval = comparison_bundle["comparison_eval"]
strategy_summary_with_external = comparison_bundle["strategy_summary"]
strategy_bucket_summary_with_external = comparison_bundle["strategy_bucket_summary"]
external_vs_cio = comparison_bundle["external_vs_cio"]
bucket_winners = comparison_bundle["bucket_winners"]
missing_external = comparison_bundle["missing_external"]

cio_row = strategy_summary_with_external[strategy_summary_with_external["strategy"] == "cio"].iloc[0]
best_short = strategy_summary_with_external.sort_values("short_accuracy", ascending=False).iloc[0]
best_long = strategy_summary_with_external.sort_values("long_accuracy", ascending=False).iloc[0]
external_text = "; ".join(
    f"{row.strategy}: short {row.short_accuracy:.1%} ({row.short_delta_vs_cio:+.1%} vs CIO), long {row.long_accuracy:.1%} ({row.long_delta_vs_cio:+.1%} vs CIO)"
    for row in external_vs_cio.itertuples(index=False)
)
missing_text = "".join(f"- Missing: `{item}`" for item in missing_external)

display(Markdown("### Overall Strategy Comparison"))
display(strategy_summary_with_external.sort_values("long_accuracy", ascending=False))

display(Markdown("### External Baselines vs CIO"))
display(external_vs_cio)

display(Markdown("### Bucket-Level Comparison"))
display(strategy_bucket_summary_with_external)

display(Markdown("### Best Strategy By Bucket"))
display(bucket_winners)

display(Markdown(f"""
### Takeaways

- CIO overall accuracy: **short {cio_row['short_accuracy']:.1%}**, **long {cio_row['long_accuracy']:.1%}**.
- Best short-horizon strategy: **{best_short.strategy}** at **{best_short.short_accuracy:.1%}**.
- Best long-horizon strategy: **{best_long.strategy}** at **{best_long.long_accuracy:.1%}**.
- External baseline comparison: {external_text if external_text else 'No external baseline files were loaded.'}
- Bucket winners show where CIO adds value versus where a single model/baseline is stronger. Use buckets with at least 3-4 cases for presentation-quality claims.

{missing_text}
"""))


### Overall Strategy Comparison

,strategy,short_evaluable,short_matched,short_accuracy,long_evaluable,long_matched,long_accuracy,short_delta_vs_cio,long_delta_vs_cio
2,market_technical,55,26,0.472727,55,31,0.563636,0.054545,0.018182
0,cio,55,23,0.418182,55,30,0.545455,0.000000,0.000000
1,news_macro,55,25,0.454545,55,29,0.527273,0.036364,-0.018182
5,gemini_baseline,55,19,0.345455,55,26,0.472727,-0.072727,-0.072727
4,gpt_baseline,55,26,0.472727,55,25,0.454545,0.054545,-0.090909
3,fundamental,55,7,0.127273,55,16,0.290909,-0.290909,-0.254545
6,claude_baseline,55,31,0.563636,55,11,0.200000,0.145455,-0.345455


### External Baselines vs CIO

,strategy,short_accuracy,short_delta_vs_cio,long_accuracy,long_delta_vs_cio,short_matched,short_evaluable,long_matched,long_evaluable
4,gpt_baseline,0.472727,0.054545,0.454545,-0.090909,26,55,25,55
5,gemini_baseline,0.345455,-0.072727,0.472727,-0.072727,19,55,26,55
6,claude_baseline,0.563636,0.145455,0.200000,-0.345455,31,55,11,55


### Bucket-Level Comparison

,bucket,cases,cio_short_accuracy,cio_long_accuracy,news_macro_short_accuracy,news_macro_long_accuracy,market_technical_short_accuracy,market_technical_long_accuracy,fundamental_short_accuracy,fundamental_long_accuracy,gpt_baseline_short_accuracy,gpt_baseline_long_accuracy,gemini_baseline_short_accuracy,gemini_baseline_long_accuracy,claude_baseline_short_accuracy,claude_baseline_long_accuracy
0,ai_infrastructure,9,0.555556,0.555556,0.555556,0.555556,0.555556,0.666667,0.111111,0.333333,0.444444,0.555556,0.666667,0.666667,0.555556,0.111111
1,ai_semis,13,0.538462,0.615385,0.461538,0.615385,0.692308,0.615385,0.153846,0.384615,0.538462,0.538462,0.076923,0.615385,0.615385,0.384615
2,data_center_operator,2,0.000000,0.500000,0.000000,0.500000,0.500000,0.500000,0.000000,0.500000,0.000000,0.500000,1.000000,0.500000,0.500000,0.000000
3,data_center_reit,4,0.250000,0.750000,0.500000,0.750000,0.250000,0.750000,0.250000,0.500000,0.500000,1.000000,0.500000,0.500000,0.500000,0.500000
4,mega_cap_platform,7,0.714286,0.285714,0.857143,0.428571,0.142857,0.142857,0.142857,0.285714,0.714286,0.285714,0.142857,0.285714,0.857143,0.285714
5,power_infrastructure,5,0.000000,0.600000,0.200000,0.600000,0.400000,0.600000,0.000000,0.200000,0.800000,0.600000,0.800000,0.600000,0.800000,0.000000
6,software_cloud,15,0.333333,0.533333,0.333333,0.400000,0.466667,0.600000,0.133333,0.133333,0.266667,0.200000,0.200000,0.266667,0.333333,0.066667


### Best Strategy By Bucket

,bucket,cases,best_short_strategy,best_short_accuracy,cio_short_accuracy,best_long_strategy,best_long_accuracy,cio_long_accuracy
0,ai_infrastructure,9,gemini_baseline,0.666667,0.555556,market_technical,0.666667,0.555556
1,ai_semis,13,market_technical,0.692308,0.538462,cio,0.615385,0.615385
2,data_center_operator,2,gemini_baseline,1.000000,0.000000,cio,0.500000,0.500000
3,data_center_reit,4,news_macro,0.500000,0.250000,gpt_baseline,1.000000,0.750000
4,mega_cap_platform,7,news_macro,0.857143,0.714286,news_macro,0.428571,0.285714
5,power_infrastructure,5,gpt_baseline,0.800000,0.000000,cio,0.600000,0.600000
6,software_cloud,15,market_technical,0.466667,0.333333,market_technical,0.600000,0.533333



### Takeaways

- CIO overall accuracy: **short 41.8%**, **long 54.5%**.
- Best short-horizon strategy: **claude_baseline** at **56.4%**.
- Best long-horizon strategy: **market_technical** at **56.4%**.
- External baseline comparison: gpt_baseline: short 47.3% (+5.5% vs CIO), long 45.5% (-9.1% vs CIO); gemini_baseline: short 34.5% (-7.3% vs CIO), long 47.3% (-7.3% vs CIO); claude_baseline: short 56.4% (+14.5% vs CIO), long 20.0% (-34.5% vs CIO)
- Bucket winners show where CIO adds value versus where a single model/baseline is stronger. Use buckets with at least 3-4 cases for presentation-quality claims.


